In [ ]:
# ── Workflow ──────────────────────────────────────────────────────────────────
# 1. Load IMDB dataset  (built-in Keras dataset, pre-tokenised)
# 2. Pad sequences      (make all reviews the same fixed length)
# 3. Build DNN model    (Embedding → Flatten → Dense layers)
# 4. Compile & Train
# 5. Evaluate accuracy
# 6. Predict sentiment on new reviews

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, Flatten, Dropout

import matplotlib.pyplot as plt

In [ ]:
# ── Step 1: Load IMDB Dataset ─────────────────────────────────────────────────
# Reviews are already tokenised: each word is replaced by an integer index.
# num_words=10000 keeps only the 10,000 most frequent words; rare words are ignored.
vocab_size = 10000

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)
print("Training samples:", len(x_train))
print("Testing samples :", len(x_test))

In [ ]:
# ── Step 2: Pad Sequences ────────────────────────────────────────────────────
# Reviews have different lengths. The Embedding layer needs a fixed input size.
# Shorter reviews are zero-padded at the start; longer ones are truncated.
max_length = 200
x_train = pad_sequences(x_train, maxlen=max_length)
x_test  = pad_sequences(x_test,  maxlen=max_length)

In [ ]:
# ── Step 3: Build the Model ───────────────────────────────────────────────────
# Embedding: turns each word index into a 32-dim vector (learned during training)
# Flatten:   collapses (200, 32) → (6400,) so Dense layers can process it
# Dropout:   randomly turns off 50% of neurons each step → reduces overfitting
# sigmoid:   outputs a probability between 0 and 1 for binary classification
model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=32, input_length=max_length))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))   # output: 0 = negative, 1 = positive

# ── Step 4: Compile ───────────────────────────────────────────────────────────
# binary_crossentropy: correct loss for a two-class problem with sigmoid output
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
# ── Step 5: Train ────────────────────────────────────────────────────────────
# batch_size=128: larger batches = faster training for this size dataset
# validation_split=0.2: 20% of training data used to monitor generalisation
history = model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2
)

In [ ]:
# ── Step 6: Evaluate ─────────────────────────────────────────────────────────
loss, accuracy = model.evaluate(x_test, y_test)
print(f"Test Loss     : {loss:.4f}")
print(f"Test Accuracy : {accuracy:.4f}")

In [ ]:
# ── Step 7: Predict on Sample Reviews ────────────────────────────────────────
# pred > 0.5 → Positive, else Negative
predictions = model.predict(x_test[:5])

print("Sample Predictions:")
for i, pred in enumerate(predictions):
    sentiment = "Positive" if pred > 0.5 else "Negative"
    print(f"  Review {i+1}: {sentiment}  (score: {pred[0]:.4f})")

In [ ]:
# ── Step 8: Plot Training Curves ─────────────────────────────────────────────
# A good model: both train & val accuracy go up and stay close together
plt.plot(history.history['accuracy'],     label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy')
plt.legend()
plt.show()

plt.plot(history.history['loss'],     label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.show()